In [1]:
# Je charge les deux fichiers de suivi avec Pandas pour travailler sur les données terrain.
# Je fixe la date du reporting au 02/10/2026 pour conserver le même périmètre que dans SQL.

import pandas as pd
import plotly.express as px

suivi_cables = pd.read_csv("suivi_cables.csv")
suivi_boites = pd.read_csv("suivi_boites.csv")
date_reporting = pd.Timestamp("2026-10-02")

print(suivi_cables.shape)
print(suivi_boites.shape)
print(suivi_boites.head())

(25033, 25)
(25033, 34)
  prestataire_id  id_pm        id_boite        id_cable  branche  section  \
0        PREST_A  PM001  PEC-PM001-1000  CDI-PM001-1000        1     1000   
1        PREST_A  PM001  PBO-PM001-1001  CDI-PM001-1001        1     1001   
2        PREST_A  PM001  PBO-PM001-1002  CDI-PM001-1002        1     1001   
3        PREST_A  PM001  PBO-PM001-1003  CDI-PM001-1003        1     1001   
4        PREST_A  PM001  PBO-PM001-1004  CDI-PM001-1004        1     1001   

   nbr_el type_boite type_support  capacite_cable_fo  ...  \
0       0        PEC      Chambre                144  ...   
1       4        PBO      Chambre                144  ...   
2      12        PBO      Chambre                144  ...   
3       6        PBO      Chambre                144  ...   
4       8    PEC-PBO      Chambre                144  ...   

  nb_replanification_recette date_pose_boite  date_reflectometrie  \
0                        0.0      2025-02-05           2025-02-14   
1       

In [2]:
# Je garde les conversions des câbles comme repère.

suivi_cables["date_debut_tirage"] = pd.to_datetime(suivi_cables["date_debut_tirage"], format="%Y-%m-%d")
suivi_cables["date_fin_tirage"] = pd.to_datetime(suivi_cables["date_fin_tirage"], format="%Y-%m-%d")
suivi_cables["date_detection_blocage"] = pd.to_datetime(suivi_cables["date_detection_blocage"], format="%Y-%m-%d")
suivi_cables["date_cloture_blocage"] = pd.to_datetime(suivi_cables["date_cloture_blocage"], format="%Y-%m-%d")

# Je prépare la liste des cinq colonnes de dates des boîtes.
colonnes_dates_boites = [
    "date_pose_boite",
    "date_reflectometrie",
    "date_recette_optique",
    "date_detection_blocage",
    "date_cloture_blocage"
]

# Je répète la même conversion pour chaque colonne de cette liste.
for nom_colonne in colonnes_dates_boites:
    suivi_boites[nom_colonne] = pd.to_datetime(suivi_boites[nom_colonne], format="%Y-%m-%d")

print(suivi_cables[["date_debut_tirage", "date_fin_tirage",
                       "date_detection_blocage", "date_cloture_blocage"]].dtypes)
print(suivi_boites[colonnes_dates_boites].dtypes)

date_debut_tirage         datetime64[ns]
date_fin_tirage           datetime64[ns]
date_detection_blocage    datetime64[ns]
date_cloture_blocage      datetime64[ns]
dtype: object
date_pose_boite           datetime64[ns]
date_reflectometrie       datetime64[ns]
date_recette_optique      datetime64[ns]
date_detection_blocage    datetime64[ns]
date_cloture_blocage      datetime64[ns]
dtype: object


In [3]:
# Je regarde les clés et les totaux avant de commencer l’analyse.
print("Câbles :", len(suivi_cables), "lignes et", suivi_cables["id_cable"].nunique(), "identifiants")
print("Boîtes :", len(suivi_boites), "lignes et", suivi_boites["id_boite"].nunique(), "identifiants")
print("Doublons câbles :", suivi_cables["id_cable"].duplicated().sum())
print("Doublons boîtes :", suivi_boites["id_boite"].duplicated().sum())
print("Clés absentes :", suivi_cables["id_cable"].isna().sum(), suivi_boites["id_boite"].isna().sum(), suivi_boites["id_cable"].isna().sum())
print("EL prévus :", suivi_boites["nbr_el"].sum())
print("PM prévus :", suivi_boites["id_pm"].nunique())
print(suivi_boites.groupby("prestataire_id")["nbr_el"].sum().to_frame("EL prévus"))

Câbles : 25033 lignes et 25033 identifiants
Boîtes : 25033 lignes et 25033 identifiants
Doublons câbles : 0
Doublons boîtes : 0
Clés absentes : 0 0 0
EL prévus : 150000
PM prévus : 450
                EL prévus
prestataire_id           
PREST_A             30000
PREST_B             30000
PREST_C             30000
PREST_D             30000
PREST_E             30000


In [4]:
# Je recherche les boîtes dont la référence câble est absente du suivi câbles.
boites_sans_cable = suivi_boites[
    ~suivi_boites["id_cable"].isin(suivi_cables["id_cable"].dropna())
]
print("Boîtes sans référence câble retrouvée :", len(boites_sans_cable))
print(boites_sans_cable[["id_boite", "id_cable"]].head(10))

# Je complète les boîtes avec le mode de pose et les dates de leur câble.
analyse_boites = suivi_boites.merge(
    suivi_cables[["id_cable", "mode_pose", "date_debut_tirage", "date_fin_tirage"]],
    on="id_cable", how="left"
)
print("Lignes après jointure :", len(analyse_boites))
print("EL après jointure :", analyse_boites["nbr_el"].sum())
print("Pose avant fin du tirage :", (analyse_boites["date_pose_boite"] < analyse_boites["date_fin_tirage"]).sum())
print("Réflectométrie avant pose :", (analyse_boites["date_reflectometrie"] < analyse_boites["date_pose_boite"]).sum())
print("Recette avant réflectométrie :", (analyse_boites["date_recette_optique"] < analyse_boites["date_reflectometrie"]).sum())
print((analyse_boites[["date_debut_tirage", "date_fin_tirage", "date_pose_boite", "date_reflectometrie", "date_recette_optique"]] > date_reporting).sum().to_frame("Dates futures"))

Boîtes sans référence câble retrouvée : 0
Empty DataFrame
Columns: [id_boite, id_cable]
Index: []
Lignes après jointure : 25033
EL après jointure : 150000
Pose avant fin du tirage : 0
Réflectométrie avant pose : 0
Recette avant réflectométrie : 0
                      Dates futures
date_debut_tirage                 0
date_fin_tirage                   0
date_pose_boite                   0
date_reflectometrie               0
date_recette_optique              0


In [5]:
# Je garde les recettes réalisées au plus tard à la date de reporting.
recettes = analyse_boites[
    analyse_boites["date_recette_optique"] <= date_reporting
].copy()
recettes["mois"] = recettes["date_recette_optique"].dt.strftime("%Y-%m")
production_mois = recettes.groupby(
    ["prestataire_id", "mois"], as_index=False
)["nbr_el"].sum()
production_mois = production_mois.rename(columns={"nbr_el": "el_recettes_mois"})
production_mois = production_mois.sort_values(["prestataire_id", "mois"])
print(production_mois.head(10))

  prestataire_id     mois  el_recettes_mois
0        PREST_A  2025-02               356
1        PREST_A  2025-03               536
2        PREST_A  2025-04               678
3        PREST_A  2025-05               956
4        PREST_A  2025-06              1016
5        PREST_A  2025-07               974
6        PREST_A  2025-08              1034
7        PREST_A  2025-09               974
8        PREST_A  2025-10              1076
9        PREST_A  2025-11               960


In [6]:
# Je cumule les livraisons depuis le début du programme.
production_mois["el_recettes_cumules"] = production_mois.groupby(
    "prestataire_id"
)["el_recettes_mois"].cumsum()
cumul_programme = production_mois.groupby("mois", as_index=False)["el_recettes_mois"].sum()
cumul_programme["el_recettes_cumules"] = cumul_programme["el_recettes_mois"].cumsum()
print("EL recettés au reporting :", recettes["nbr_el"].sum())
print(recettes.groupby("prestataire_id")["nbr_el"].sum().to_frame("Cumul des EL recettés"))
print(cumul_programme)

EL recettés au reporting : 87294
                Cumul des EL recettés
prestataire_id                       
PREST_A                         17868
PREST_B                         17436
PREST_C                         17654
PREST_D                         17186
PREST_E                         17150
       mois  el_recettes_mois  el_recettes_cumules
0   2025-02              1342                 1342
1   2025-03              2244                 3586
2   2025-04              3128                 6714
3   2025-05              4148                10862
4   2025-06              4722                15584
5   2025-07              4784                20368
6   2025-08              4898                25266
7   2025-09              4822                30088
8   2025-10              4936                35024
9   2025-11              4810                39834
10  2025-12              4900                44734
11  2026-01              4682                49416
12  2026-02              4894         

In [7]:
# Je prépare une liste avec une seule ligne par prestataire du programme.
objectif_mensuel = 850
prestataires_attendus = suivi_boites[["prestataire_id"]].dropna().drop_duplicates()
prestataires_attendus = prestataires_attendus.sort_values("prestataire_id")
print("Prestataires présents dans le programme :", len(prestataires_attendus), "/ 5")
production_complete = production_mois[
    (production_mois["mois"] >= "2025-02") & (production_mois["mois"] < "2026-10")
].copy()

# Je rattache les mois observés à cette liste avec une jointure à gauche.
mois_observes = production_complete.groupby("prestataire_id", as_index=False)["mois"].nunique()
mois_observes = mois_observes.rename(columns={"mois": "nb_mois_observes"})
mois_observes = prestataires_attendus.merge(mois_observes, on="prestataire_id", how="left")
mois_observes["nb_mois_observes"] = mois_observes["nb_mois_observes"].fillna(0)
print("Mois complets observés par prestataire, sur 20 :")
print(mois_observes)
print("Prestataires avec des mois absents :")
print(mois_observes[mois_observes["nb_mois_observes"] < 20])

# Je garde aussi tous les prestataires dans le bilan de septembre.
bilan_mois = production_complete[production_complete["mois"] == "2026-09"].copy()
bilan_mois = prestataires_attendus.merge(bilan_mois, on="prestataire_id", how="left")
bilan_mois["mois"] = "2026-09"
bilan_mois["el_recettes_mois"] = bilan_mois["el_recettes_mois"].fillna(0)
bilan_mois["objectif_el"] = objectif_mensuel
bilan_mois["ecart_el"] = bilan_mois["el_recettes_mois"] - objectif_mensuel
bilan_mois["atteinte_objectif"] = bilan_mois["el_recettes_mois"] / objectif_mensuel
bilan_affiche = bilan_mois.copy()
bilan_affiche["atteinte_objectif_pct"] = bilan_affiche["atteinte_objectif"] * 100
print(bilan_affiche[["prestataire_id", "el_recettes_mois", "objectif_el", "ecart_el",
                      "atteinte_objectif_pct"]].round(1))
print("Prestataires à l’objectif :", (bilan_mois["ecart_el"] >= 0).sum(),
      "/", len(prestataires_attendus))

Prestataires présents dans le programme : 5 / 5
Mois complets observés par prestataire, sur 20 :
  prestataire_id  nb_mois_observes
0        PREST_A                20
1        PREST_B                20
2        PREST_C                20
3        PREST_D                20
4        PREST_E                20
Prestataires avec des mois absents :
Empty DataFrame
Columns: [prestataire_id, nb_mois_observes]
Index: []
  prestataire_id  el_recettes_mois  objectif_el  ecart_el  \
0        PREST_A               896          850        46   
1        PREST_B               944          850        94   
2        PREST_C               900          850        50   
3        PREST_D               944          850        94   
4        PREST_E               930          850        80   

   atteinte_objectif_pct  
0                  105.4  
1                  111.1  
2                  105.9  
3                  111.1  
4                  109.4  
Prestataires à l’objectif : 5 / 5


In [8]:
# Je montre l’évolution des livraisons et l’objectif mensuel.
fig_mois = px.line(
    production_complete, x="mois", y="el_recettes_mois",
    color="prestataire_id", markers=True,
    labels={"mois": "Mois", "el_recettes_mois": "EL recettés",
            "prestataire_id": "Prestataire"},
    title="EL recettés par mois — février 2025 à septembre 2026"
)
fig_mois.add_hline(y=850, line_dash="dash", annotation_text="Objectif : 850 EL par prestataire")
fig_mois.update_layout(template="plotly_white", font_size=14, height=550)
fig_mois.update_xaxes(type="category", tickangle=-45)
fig_mois.show()

In [9]:
# Je garde les câbles dont le tirage est terminé à la date du reporting.
cables_tires = suivi_cables[
    suivi_cables["date_fin_tirage"] <= date_reporting
].copy()

print("Câbles tirés :", len(cables_tires))
print("Câbles prévus :", len(suivi_cables))
taux_cables_tires = len(cables_tires) / len(suivi_cables) * 100
print("Part des câbles tirés (%) :", round(taux_cables_tires, 1))

Câbles tirés : 15438
Câbles prévus : 25033
Part des câbles tirés (%) : 61.7


In [10]:
# Je compte les câbles terminés par mois pour observer le rythme de tirage.
cables_tires["mois"] = cables_tires["date_fin_tirage"].dt.strftime("%Y-%m")
tirages_mois = cables_tires.groupby("mois")["id_cable"].count()
tirages_mois = tirages_mois.reset_index(name="nb_cables_tires")
tirages_mois = tirages_mois.sort_values("mois")
print(tirages_mois)
print("Total des tirages mensuels :", tirages_mois["nb_cables_tires"].sum())

# Je trace uniquement les mois complets pour comparer des périodes comparables.
tirages_complets = tirages_mois[
    (tirages_mois["mois"] >= "2025-02") & (tirages_mois["mois"] < "2026-10")
].copy()
print("Mois de tirage observés :", tirages_complets["mois"].nunique(), "/ 20")
fig_tirages = px.line(
    tirages_complets, x="mois", y="nb_cables_tires", markers=True,
    title="Câbles tirés par mois — mois complets",
    labels={"mois": "Mois", "nb_cables_tires": "Nombre de câbles tirés"}
)
fig_tirages.update_layout(template="plotly_white", font_size=14, height=500)
fig_tirages.update_xaxes(type="category", tickangle=-45)
fig_tirages.show()

       mois  nb_cables_tires
0   2025-02              417
1   2025-03              426
2   2025-04              592
3   2025-05              783
4   2025-06              704
5   2025-07              816
6   2025-08              850
7   2025-09              764
8   2025-10              854
9   2025-11              777
10  2025-12              813
11  2026-01              807
12  2026-02              770
13  2026-03              753
14  2026-04              837
15  2026-05              802
16  2026-06              745
17  2026-07              807
18  2026-08              802
19  2026-09             1160
20  2026-10              159
Total des tirages mensuels : 15438
Mois de tirage observés : 20 / 20


In [11]:
# Je vérifie les métrages avant de comparer les longueurs sur les mêmes câbles.
print("Métrages manquants par colonne :")
print(cables_tires[["valeur_debut", "valeur_fin", "longueur_cartographique_m"]].isna().sum())
cables_tires["longueur_reelle_m"] = cables_tires["valeur_fin"] - cables_tires["valeur_debut"]
print("Longueurs réelles non positives :", (cables_tires["longueur_reelle_m"] <= 0).sum())
print("Longueurs cartographiques non positives :",
      (cables_tires["longueur_cartographique_m"] <= 0).sum())

# Je garde uniquement les câbles avec deux longueurs renseignées et positives.
cables_mesurables = cables_tires.dropna(
    subset=["valeur_debut", "valeur_fin", "longueur_cartographique_m"]
).copy()
cables_mesurables = cables_mesurables[
    (cables_mesurables["longueur_reelle_m"] > 0)
    & (cables_mesurables["longueur_cartographique_m"] > 0)
].copy()
print("Câbles exclus du comparatif :", len(cables_tires) - len(cables_mesurables))
print("Câbles comparés :", len(cables_mesurables), "/", len(cables_tires), "tirés")
longueurs = cables_mesurables.groupby("mode_pose")[
    ["longueur_reelle_m", "longueur_cartographique_m"]
].sum()
longueurs["ecart_longueur"] = (
    longueurs["longueur_reelle_m"] - longueurs["longueur_cartographique_m"]
) / longueurs["longueur_cartographique_m"]
longueurs = longueurs.reset_index()
longueurs_affiche = longueurs.copy()
longueurs_affiche["ecart_longueur_pct"] = longueurs_affiche["ecart_longueur"] * 100
print(longueurs_affiche[["mode_pose", "longueur_reelle_m", "longueur_cartographique_m",
                          "ecart_longueur_pct"]].round(2))

Métrages manquants par colonne :
valeur_debut                 0
valeur_fin                   0
longueur_cartographique_m    0
dtype: int64
Longueurs réelles non positives : 0
Longueurs cartographiques non positives : 0
Câbles exclus du comparatif : 0
Câbles comparés : 15438 / 15438 tirés
    mode_pose  longueur_reelle_m  longueur_cartographique_m  \
0      Aerien           795598.2                   725828.9   
1      Façade           306495.8                   280364.1   
2    Immeuble           223391.1                   203621.2   
3  Souterrain           967250.6                   882099.7   

   ecart_longueur_pct  
0                9.61  
1                9.32  
2                9.71  
3                9.65  


In [12]:
# Je compare visuellement les longueurs cartographiques et les longueurs tirées.
fig_longueurs = px.bar(
    longueurs, x="mode_pose",
    y=["longueur_cartographique_m", "longueur_reelle_m"],
    barmode="group",
    title="Longueurs des câbles tirés — métrages valides",
    labels={"mode_pose": "Mode de pose", "value": "Longueur (m)",
            "variable": "Type de longueur"}
)
fig_longueurs.update_layout(template="plotly_white", font_size=14, height=500)
fig_longueurs.show()

In [13]:
# Je sélectionne les blocages ouverts et je les classe par cause.
blocages_boites = analyse_boites[
    (analyse_boites["date_detection_blocage"] <= date_reporting)
    & (analyse_boites["date_cloture_blocage"].isna()
       | (analyse_boites["date_cloture_blocage"] > date_reporting))
].copy()
blocages_boites["anciennete_jours"] = (
    date_reporting - blocages_boites["date_detection_blocage"]
).dt.days
print(blocages_boites["anciennete_jours"].describe().to_frame("Ancienneté en jours"))
# Je garde les causes absentes dans une catégorie explicite.
blocages_boites["cause_blocage"] = (
    blocages_boites["cause_blocage"].fillna("Non renseignée").str.strip()
    .replace("", "Non renseignée")
)
causes_boites = blocages_boites.groupby("cause_blocage")["id_boite"].count()
causes_boites = causes_boites.reset_index(name="nb_boites_bloquees")
causes_boites = causes_boites.sort_values("nb_boites_bloquees", ascending=True)
print("Boîtes bloquées au total :", len(blocages_boites))
print("Boîtes comptées par cause :", causes_boites["nb_boites_bloquees"].sum())
print("Écart de rapprochement :",
      len(blocages_boites) - causes_boites["nb_boites_bloquees"].sum())
fig_causes = px.bar(causes_boites, x="nb_boites_bloquees", y="cause_blocage",
    orientation="h", text="nb_boites_bloquees", title="Boîtes bloquées par cause — 02/10/2026",
    labels={"nb_boites_bloquees": "Nombre de boîtes bloquées", "cause_blocage": "Cause"})
fig_causes.update_layout(template="plotly_white", font_size=14, height=650)
fig_causes.show()

       Ancienneté en jours
count           189.000000
mean              7.015873
std               7.340123
min               1.000000
25%               2.000000
50%               4.000000
75%              10.000000
max              38.000000
Boîtes bloquées au total : 189
Boîtes comptées par cause : 189
Écart de rapprochement : 0


In [14]:
# J’enregistre mes résultats dans six fichiers CSV séparés.
production_mois.to_csv("python_controle_production_mensuelle.csv", index=False, encoding="utf-8-sig")
cumul_programme.to_csv("python_cumul_el_recettes.csv", index=False, encoding="utf-8-sig")
bilan_mois.to_csv("python_objectif_septembre.csv", index=False, encoding="utf-8-sig")
tirages_mois.to_csv("python_cables_tires_par_mois.csv", index=False, encoding="utf-8-sig")
longueurs.to_csv("python_controle_longueurs.csv", index=False, encoding="utf-8-sig")
causes_boites.to_csv("python_controle_causes_boites.csv", index=False, encoding="utf-8-sig")

print("Les six CSV sont disponibles dans le panneau Fichiers de Colab.")

Les six CSV sont disponibles dans le panneau Fichiers de Colab.
